In [ ]:
# ============================================================
# ASSIGNMENT 2 - MIND LEADERBOARD SUBMISSION (LightGBM, full test)
# Team: Shivam Patel + Suyash Pande | CS4.406 IRE
#
# Loads the trained A2 LightGBM reranker (from HF, no retraining) and scores the FULL
# MINDlarge_test (2.37M impressions, unlabeled). Builds the SAME 14 Q1 features per
# candidate (point-in-time, leakage-safe), ranks each slate, writes Codabench format:
#     {impression_id} [rank1,rank2,...]     (rank = position of each candidate, 1=top)
# Streams predictions to file (memory-safe for 2.37M impressions), zips, ready to submit.
#
# HF donbosoc/mind-artifacts: download WITHOUT token.
# NEEDS: Kaggle MIND-large mounted (train for feature stats + test for prediction).
# Submit prediction.zip to Codabench MIND competition 13967.
# ============================================================
!pip install lightgbm huggingface_hub sentence-transformers -q
import os, glob, re, math, zipfile, numpy as np, pandas as pd, datetime as dt
from bisect import bisect_left
from collections import defaultdict, Counter
import lightgbm as lgb
from huggingface_hub import hf_hub_download
HF_REPO="donbosoc/mind-artifacts"
def dl(name): return hf_hub_download(HF_REPO, name, repo_type="dataset")

LTR=glob.glob("/kaggle/input/**/MINDlarge_train",recursive=True)[0]
LDV=glob.glob("/kaggle/input/**/MINDlarge_dev",recursive=True)[0]
LTE=glob.glob("/kaggle/input/**/MINDlarge_test",recursive=True)[0]
print("train:",LTR,"\ndev:",LDV,"\ntest:",LTE)
NEWS=["news_id","category","subcategory","title","abstract","url","te","ae"]
BEH =["impression_id","user_id","time","history","impressions"]
_WORD=re.compile(r"[^\W\d_]+", re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if isinstance(t,str) else []
def pfx(x): return f"mind:{x}"
MAX_HIST=50


In [ ]:
# ---- catalogue (train+dev+TEST) + MiniLM embeddings ----
news=pd.concat([pd.read_csv(f"{d}/news.tsv",sep="\t",header=None,names=NEWS,quoting=3,
               usecols=["news_id","category","title","abstract"]) for d in (LTR,LDV,LTE)]
              ).drop_duplicates("news_id").reset_index(drop=True)
news["title"]=news["title"].fillna(""); news["abstract"]=news["abstract"].fillna("")
ids=[pfx(r.news_id) for r in news.itertuples()]
cat_lut={pfx(r.news_id):(r.category if isinstance(r.category,str) else "") for r in news.itertuples()}
from sentence_transformers import SentenceTransformer
minilm=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb=minilm.encode([f"{r.title} {r.abstract}".strip() for r in news.itertuples()],
                  batch_size=512,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
emb_by_id={ids[i]:emb[i] for i in range(len(ids))}
print("catalogue:",len(ids),"| emb:",emb.shape)


In [ ]:
# ---- point-in-time event index from train+dev+TEST impressions ----
# TEST is unlabeled (no click labels) but its IMPRESSION events still inform popularity/CTR
# denominators and first-seen; clicks come only from train+dev. All strictly-before-T.
def load_beh(p):
    b=pd.read_csv(f"{p}/behaviors.tsv",sep="\t",header=None,names=BEH,quoting=3)
    b["t"]=pd.to_datetime(b["time"],format="%m/%d/%Y %I:%M:%S %p",errors="coerce"); return b
print("loading behaviors...")
b_tr=load_beh(LTR); b_dv=load_beh(LDV); b_te=load_beh(LTE)
print("train",len(b_tr),"dev",len(b_dv),"test",len(b_te))

hist_lut={}
for b in (b_tr,b_dv,b_te):
    for u,h in zip(b["user_id"],b["history"]):
        if isinstance(h,str) and h: hist_lut[pfx(u)]=[pfx(x) for x in h.split()]

click_ev=defaultdict(list); imp_ev=defaultdict(list); first_seen={}
for b in (b_tr,b_dv,b_te):
    for t,imps in zip(b["t"],b["impressions"]):
        if pd.isna(t) or not isinstance(imps,str): continue
        for tkn in imps.split():
            p=tkn.split("-"); nid=pfx(p[0])
            if nid not in first_seen or t<first_seen[nid]: first_seen[nid]=t
            imp_ev[nid].append(t)                      # test tokens have no -label
            if len(p)==2 and p[1]=="1": click_ev[nid].append(t)
for d in (click_ev,imp_ev):
    for k in d: d[k].sort()
print("users w/ history:",len(hist_lut),"| articles w/ clicks:",len(click_ev))


In [ ]:
# ---- feature functions (identical to A2_01, leakage-safe) ----
def cnt(d,aid,T,w=None):
    tl=d.get(aid)
    if not tl: return 0
    hi=bisect_left(tl,T); return hi if w is None else hi-bisect_left(tl,T-dt.timedelta(hours=w))
def ctr(aid,T,w=None):
    c=cnt(click_ev,aid,T,w); s=cnt(imp_ev,aid,T,w); return c/s if s>0 else 0.0
def freshness(aid,T,tau=6.0):
    fs=first_seen.get(aid)
    if fs is None or T is None: return 0.0
    dh=(T-fs).total_seconds()/3600.0
    return float(np.exp(-dh/tau)) if dh>=0 else 0.0
def recency_wt_hist(uid,mh=50,tau=8.0):
    ai=hist_lut.get(uid,[])[-mh:]; vs=[emb_by_id[a] for a in ai if a in emb_by_id]
    if not vs: return None
    nn_=len(vs); w=np.exp(-(np.arange(nn_)[::-1])/tau); w/=w.sum()
    um=np.average(np.array(vs),axis=0,weights=w); return um/(np.linalg.norm(um)+1e-9)
def user_cat_dist(uid,mh=50):
    ai=hist_lut.get(uid,[])[-mh:]; cats=[cat_lut.get(a) for a in ai]
    tot=len([c for c in cats if c]); cc=Counter(c for c in cats if c)
    return {k:v/tot for k,v in cc.items()} if tot else {}
def mm(x):
    lo,hi=x.min(),x.max(); return np.zeros_like(x) if hi-lo<1e-12 else (x-lo)/(hi-lo)

FEAT=["hist_emb_mean","hist_emb_best","recency_wt_sim","click_count","pop_1h","pop_24h",
      "pop_7d","pop_total","ctr_24h","ctr_total","freshness","cat_match","position","slate_size"]
def build_feats(uid,T,cand):
    hv=[emb_by_id[a] for a in hist_lut.get(uid,[])[-50:] if a in emb_by_id]
    um=(np.mean(hv,0)/(np.linalg.norm(np.mean(hv,0))+1e-9)) if hv else None
    rw=recency_wt_hist(uid); uc=user_cat_dist(uid); hlen=len(hist_lut.get(uid,[])); m=len(cand); F=[]
    for i,c in enumerate(cand):
        cv=emb_by_id.get(c)
        em=float(um@cv) if (um is not None and cv is not None) else 0.0
        eb=float(max((v@cv for v in hv),default=0.0)) if cv is not None else 0.0
        rws=float(rw@cv) if (rw is not None and cv is not None) else 0.0
        p1=cnt(click_ev,c,T,1);p24=cnt(click_ev,c,T,24);p7=cnt(click_ev,c,T,168);ptot=cnt(click_ev,c,T)
        c24=ctr(c,T,24);ctot=ctr(c,T);fr=freshness(c,T);cm=uc.get(cat_lut.get(c,""),0.0);pos=i/max(1,m-1)
        F.append([em,eb,rws,hlen,p1,p24,p7,ptot,c24,ctot,fr,cm,pos,m])
    F=np.array(F,float)
    for col in [4,5,6,7]: F[:,col]=mm(F[:,col])
    return F
print("feature builder ready")


In [ ]:
# ---- load trained LightGBM + predict on FULL test, streaming to file ----
lgbm=lgb.Booster(model_file=dl("mind_lgbm_reranker.txt")); print("loaded LightGBM")

import time
out_path="/kaggle/working/prediction.txt"
t0=time.time(); n=0
with open(out_path,"w") as fout:
    for row in b_te.itertuples():
        if not isinstance(row.impressions,str): continue
        cand=[pfx(tk.split("-")[0]) for tk in row.impressions.split()]  # test tokens: id only (no -label)
        T=row.t
        F=build_feats(pfx(row.user_id), T, cand)
        sc=lgbm.predict(F)
        order=np.argsort(-sc)
        ranks=np.empty(len(cand),int); ranks[order]=np.arange(1,len(cand)+1)
        fout.write(f"{row.impression_id} [{','.join(map(str,ranks.tolist()))}]\n")
        n+=1
        if n%100000==0:
            el=time.time()-t0; print(f"  {n}/{len(b_te)}  ({el/60:.1f} min, ~{n/el:.0f} imp/s)")
print(f"wrote {n} predictions to {out_path} in {(time.time()-t0)/60:.1f} min")


In [ ]:
# ---- zip for Codabench + sanity check format ----
with zipfile.ZipFile("/kaggle/working/prediction.zip","w",zipfile.ZIP_DEFLATED) as z:
    z.write("/kaggle/working/prediction.txt","prediction.txt")
# show first 3 lines to verify format: "{impression_id} [r1,r2,...]"
with open("/kaggle/working/prediction.txt") as f:
    for _ in range(3): print(f.readline().strip())
print("\nprediction.zip ready -> submit to Codabench MIND competition 13967")
print("(format: impression_id [ranks], rank=position of each candidate, 1=top)")
